# Test a vLLM endpoint with the OpenAI SDK

This notebook connects to a running Nebius endpoint that serves a base model and a LoRA adapter through vLLM.

You will inspect the available models, compare base and adapter answers on held-out examples, try custom prompts, and continue a short conversation.

No system message is added, matching the public-domain character training setup.


## 1. Install and import the SDK

In [ ]:
# %pip install -q -r requirements.txt


In [10]:
from pathlib import Path
import json
import os
import re
import time

from IPython.display import Markdown, display
from openai import APIConnectionError, InternalServerError, OpenAI


## 2. Connect to the endpoint

Paste the public endpoint URL. The OpenAI client requires an API key string, so use any non-empty placeholder when endpoint authentication is disabled.


In [11]:
ENDPOINT_URL = os.getenv("ENDPOINT_URL", "https://port8000-sb5rec4tjrsft17.tunnel.applications.eu-north1.nebius.cloud").rstrip("/")
API_KEY = os.getenv("ENDPOINT_API_KEY", "not-needed")

client = OpenAI(
    base_url=f"{ENDPOINT_URL}/v1",
    api_key=API_KEY,
)

print("Endpoint:", ENDPOINT_URL)


Endpoint: https://port8000-sb5rec4tjrsft17.tunnel.applications.eu-north1.nebius.cloud


## 3. Inspect the models served by vLLM

The response normally contains one base model and one LoRA adapter. The adapter has the base model in its parent field.


In [12]:
def wait_for_models(timeout=600, poll_interval=10):
    deadline = time.monotonic() + timeout

    while True:
        try:
            return [model.model_dump() for model in client.models.list().data]
        except (APIConnectionError, InternalServerError) as error:
            if time.monotonic() >= deadline:
                raise
            print(
                "Endpoint is running but vLLM is not ready yet. "
                f"Retrying in {poll_interval} seconds..."
            )
            time.sleep(poll_interval)


served_models = wait_for_models()

for model in served_models:
    print(f"id={model['id']!r}  parent={model.get('parent')!r}")


id='Qwen/Qwen2.5-7B-Instruct'  parent=None
id='demo'  parent='Qwen/Qwen2.5-7B-Instruct'


In [13]:
base_candidates = [
    model["id"] for model in served_models
    if model.get("parent") is None
]
adapter_candidates = [
    model["id"] for model in served_models
    if model.get("parent") is not None
]

if not base_candidates:
    raise ValueError("No base model found")
if not adapter_candidates:
    raise ValueError("No LoRA adapter found")

# Override these values if the endpoint serves multiple models or adapters.
BASE_MODEL = base_candidates[0]
ADAPTER_MODEL = adapter_candidates[0]

print("Base model:   ", BASE_MODEL)
print("LoRA adapter:", ADAPTER_MODEL)


Base model:    Qwen/Qwen2.5-7B-Instruct
LoRA adapter: demo


## 4. Send a chat completion

Messages use the familiar OpenAI format: a list containing role and content.


In [24]:
def chat(model, messages, temperature=0.8, max_tokens=160):
    completion = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return completion.choices[0].message.content


example_messages = [
    {
        "role": "user",
        "content": "What quality do you value most in a trusted companion?",
    }
]

print(chat(ADAPTER_MODEL, example_messages))


In a companion? why, that he be of the very whey and consistence of my friend; that he be precise as I am, loving as I am, valiant as I am, wise as I am.


## 6. Main comparison: custom neutral prompts

These prompts do not name or describe any character and contain no previous assistant answer. Edit the list or add your own questions. This is the cleanest way to observe what style the adapter learned.


In [31]:
def compare_messages(messages, temperature=0.2):
    # The same list of messages is used in both API calls.
    return {
        "prompt": messages,
        "base": chat(BASE_MODEL, messages, temperature=temperature),
        "adapter": chat(ADAPTER_MODEL, messages, temperature=temperature),
    }


def show_comparison(result, title):
    separator = chr(10) * 2
    prompt_text = separator.join(
        f"**{message['role'].title()}:** {message['content']}"
        for message in result["prompt"]
    )
    markdown = separator.join([
        f"### {title}",
        prompt_text,
        f"**Base model:** {result['base']}",
        f"**LoRA adapter:** {result['adapter']}",
    ])
    display(Markdown(markdown))


In [33]:
custom_prompts = [
    "What do you consider the greatest weakness in human nature?",
    "Describe the strangest encounter you have ever had to a curious traveller.",
    "What counsel would you offer before I enter a place said to be dangerous?",
]

for number, prompt_text in enumerate(custom_prompts, start=1):
    messages = [{"role": "user", "content": prompt_text}]
    result = compare_messages(messages, temperature=0.5)
    show_comparison(result, f"Custom prompt {number}")


### Custom prompt 1

**User:** What do you consider the greatest weakness in human nature?

**Base model:** The concept of "greatest weakness in human nature" can be subjective and varies depending on cultural, philosophical, and individual perspectives. However, many people and scholars have identified several common weaknesses that are often discussed:

1. **Selfishness**: The tendency to prioritize one's own interests over others can lead to conflicts and a lack of cooperation.

2. **Pride**: Excessive pride or arrogance can cloud judgment and prevent individuals from acknowledging their mistakes or learning from others.

3. **Insecurity**: Constant fear of rejection, failure, or inadequacy can drive irrational behavior and hinder personal growth.

4. **Laziness**: A tendency to avoid effort or responsibility can impede progress and achievement.

5. **Short-sightedness**: Focusing only on immediate gratification rather than long

**LoRA adapter:** The greatest weakness that I can find in human nature is the want of patience.

### Custom prompt 2

**User:** Describe the strangest encounter you have ever had to a curious traveller.

**Base model:** As an artificial intelligence, I don't have personal experiences or encounters. However, I can certainly create a fictional story that might intrigue a curious traveler! Here's a tale that could fit the bill:

---

In the heart of the Amazon rainforest, under the dense canopy where sunlight barely pierced through, I found myself on a journey that was both thrilling and bewildering. My guide, a local shaman named Kael, led me through a path less trodden, whispering about ancient spirits and forgotten rituals.

After hours of walking, we stumbled upon a clearing. In the center stood a large stone altar, covered in intricate carvings that seemed to glow faintly in the dim light. Kael explained that this was the site of an ancient ritual, one that had been performed for generations

**LoRA adapter:** I'll tell you what, sir: I had rather be a country justice, as they call it, than a king; for, in my poor judgment, there is no spot so bad in a kingdom but the justice of the country will make it better than any spot that's good in a bad kingdom.

### Custom prompt 3

**User:** What counsel would you offer before I enter a place said to be dangerous?

**Base model:** Entering a potentially dangerous area requires careful planning and preparation. Here are some general pieces of advice that can help ensure your safety:

1. **Research Thoroughly**: Gather as much information as possible about the area. Understand the risks involved and any local laws or customs.

2. **Stay Informed**: Keep up-to-date with the latest news and alerts from reliable sources. This can provide crucial information about any ongoing threats or changes in the environment.

3. **Travel in Groups**: If possible, travel with others. Having a group can deter potential threats and provide mutual support.

4. **Inform Someone About Your Plans**: Let someone know where you are going, who you are with, and when you expect to return. This can help in case of an emergency.

5. **Pack Appropriately

**LoRA adapter:** That you would not enter where you were most sure to die.

## 7. Continue a conversation

For a follow-up, append the assistant answer and the next user message to the history, then send the full list again. Change CHAT_MODEL to BASE_MODEL to try the same conversation with the base model.


In [ ]:
CHAT_MODEL = ADAPTER_MODEL

conversation = [
    {
        "role": "user",
        "content": "A locked door stands before us, and there is laughter behind it. What should we do?",
    }
]

first_answer = chat(CHAT_MODEL, conversation, temperature=0.7)
conversation.append({"role": "assistant", "content": first_answer})
print("Assistant:", first_answer)

conversation.append({
    "role": "user",
    "content": "And what makes you so certain that your advice is sound?",
})
second_answer = chat(CHAT_MODEL, conversation, temperature=0.7)
conversation.append({"role": "assistant", "content": second_answer})
print()
print("Assistant:", second_answer)
